# 🤖 AI Engineering Fundamentals — Lezione 2
## Notebook Gruppo A

**ITS Novitas 4.0 | Giovedì 21/05/2026**

---

### 📋 Istruzioni
1. **File → Salva una copia in Drive** prima di iniziare
2. Lavorate in gruppo — discutete prima di scrivere
3. Alla fine: **File → Scarica → .ipynb** e caricate su GitHub

### 👥 Membri del gruppo

In [ ]:
GRUPPO = "A"
MEMBRI = ["", "", "", ""]  # ← inserite i vostri nomi
print(f"Gruppo {GRUPPO} — {', '.join(m for m in MEMBRI if m)}")

In [ ]:
# Setup — eseguite questa cella per prima
# La API key viene letta dal file .env nella root del progetto (non più dai Secrets di Colab)
import anthropic
from dotenv import load_dotenv, find_dotenv

load_dotenv(find_dotenv(usecwd=True))   # carica ANTHROPIC_API_KEY dal .env

client = anthropic.Anthropic()          # legge automaticamente ANTHROPIC_API_KEY dall'ambiente

def chiedi_claude(messaggio, temperature=0.7, system=None, max_tokens=600):
    params = {
        "model": "claude-haiku-4-5-20251001",
        "max_tokens": max_tokens,
        "temperature": temperature,
        "messages": [{"role": "user", "content": messaggio}]
    }
    if system:
        params["system"] = system
    return client.messages.create(**params).content[0].text

print("✅ Setup completato!")

---
## 🎯 Tema del Gruppo A: Zero-shot, Few-shot & Chain-of-Thought

Esplorate le tre tecniche fondamentali del prompt engineering
e scoprite quando usare ciascuna.

---
### Esercizio 1 — Zero-shot vs Few-shot *(guidato)*

Classificate il sentiment di recensioni prima senza esempi,
poi con esempi. Confrontate precisione e formato.

In [ ]:
recensioni = [
    "Il sensore XS200 funziona perfettamente, installazione semplicissima!",
    "Prodotto nella media, niente di eccezionale ma fa il suo lavoro.",
    "Pessima esperienza: il gateway si è disconnesso dopo 2 giorni.",
    "Dashboard Xplore un po' lenta ma i dati sono accurati.",
]

# ZERO-SHOT: nessun esempio
print("=" * 55)
print("ZERO-SHOT")
print("=" * 55)

prompt_zs = "Classifica il sentiment di questa recensione come POSITIVO, NEGATIVO o NEUTRO:\n\n"

for rec in recensioni:
    # Classificazione: temperature=0 per risposte stabili
    r = chiedi_claude(rec, temperature=0, system=prompt_zs)
    print(f"Recensione: '{rec[:50]}...'")
    print(f"Sentiment:  {r}\n")

print()

# FEW-SHOT: con 3 esempi
print("=" * 55)
print("FEW-SHOT (3 esempi)")
print("=" * 55)

prompt_fs = """Classifica il sentiment di recensioni. Esempi:

Recensione: "Ottimo prodotto, lo ricompro!"       → POSITIVO
Recensione: "Non funziona, sono molto deluso."    → NEGATIVO
Recensione: "Consegnato in 3 giorni lavorativi." → NEUTRO

Ora classifica questa recensione con UNA sola parola (POSITIVO/NEGATIVO/NEUTRO):

"""

for rec in recensioni:
    # prompt_fs + rec come unico messaggio, temperature=0
    r = chiedi_claude(prompt_fs + rec, temperature=0)
    print(f"Recensione: '{rec[:50]}...'")
    print(f"Sentiment:  {r}\n")

# Osservazione: con lo zero-shot Claude tende a rispondere in modo prolisso
# (spiega la classificazione). Con il few-shot impara dal formato degli esempi
# e risponde con UNA sola parola: molto più affidabile da elaborare a valle.

---
### Esercizio 2 — Chain-of-Thought *(guidato)*

Stesso problema con e senza "pensa passo per passo".
Quanto cambia la qualità della risposta?

In [ ]:
problema = """
WiData ha installato sensori in 3 città sarde:
- Sassari: 12 sensori, costo €45 ciascuno
- Cagliari: 8 sensori, costo €45 ciascuno
- Nuoro: 5 sensori, costo €38 ciascuno (modello più vecchio)

Il comune di Olbia vuole installare sensori per un budget di €800.
Quanti sensori del modello nuovo può comprare?
E quanti del modello vecchio?
"""

# SENZA Chain-of-Thought
print("=" * 55)
print("SENZA Chain-of-Thought:")
print("=" * 55)
print(chiedi_claude(problema, temperature=0))

print()

# CON Chain-of-Thought
print("=" * 55)
print("CON Chain-of-Thought:")
print("=" * 55)
problema_cot = problema + "\nPensa passo per passo e mostra tutti i calcoli prima di dare la risposta finale."
print(chiedi_claude(problema_cot, temperature=0))

# Osservazione: chiedendo di "pensare passo per passo" il modello mostra i calcoli
# (es. 800 / 45 = 17 sensori nuovi; 800 / 38 = 21 sensori vecchi) e arriva più
# spesso al risultato corretto. Senza CoT può saltare passaggi e sbagliare i conti.

---
### Esercizio 3 — Quanti esempi servono nel few-shot? *(libero)*

Testate lo stesso task di classificazione con 1, 3 e 5 esempi.
C'è un punto oltre cui aggiungere esempi non migliora più la risposta?

In [ ]:
# Esercizio 3 — 1-shot vs 3-shot vs 5-shot
# Usate lo stesso task di classificazione dell'esercizio 1

# Esempi disponibili (usatene 1, poi 3, poi tutti 5)
esempi = [
    ('"Prodotto eccellente, supera le aspettative!"', 'POSITIVO'),
    ('"Non funziona come descritto, deluso."', 'NEGATIVO'),
    ('"Spedizione veloce, imballaggio standard."', 'NEUTRO'),
    ('"Qualità superiore, lo consiglio vivamente."', 'POSITIVO'),
    ('"Reso facile, assistenza nella media."', 'NEUTRO'),
]

recensione_test = "Il sensore funziona ma l'app è difficile da configurare."

def prompt_con_n_esempi(n):
    """Costruisce un prompt few-shot usando i primi n esempi."""
    testo = "Classifica il sentiment delle recensioni con UNA sola parola (POSITIVO/NEGATIVO/NEUTRO). Esempi:\n\n"
    for rec, label in esempi[:n]:
        testo += f"Recensione: {rec} → {label}\n"
    testo += f'\nOra classifica questa recensione:\nRecensione: "{recensione_test}" →'
    return testo

for n in [1, 3, 5]:
    risposta = chiedi_claude(prompt_con_n_esempi(n), temperature=0, max_tokens=10)
    print(f"{n}-shot → {risposta.strip()}")

# Conclusione:
# 1-shot → di solito già classifica correttamente, ma il formato può variare un po'
# 3-shot → formato stabile (una parola) e classificazione corretta
# 5-shot → risultato uguale al 3-shot
# Aggiungere esempi aiuta fino a un certo punto: dopo 3-4 esempi ben scelti il
# miglioramento è marginale, mentre i token (e il costo) continuano a crescere.

---
### Esercizio 4 — CoT su un problema logico *(libero)*

Inventate un problema logico (non matematico) legato a WiData
— ad esempio una decisione su quale sensore installare in base
a certi requisiti. Testate con e senza CoT.

Il CoT aiuta anche su problemi non matematici?

In [ ]:
# Esercizio 4 — CoT su problema logico

mio_problema = """
Un cliente WiData vuole monitorare 3 capannoni industriali e deve scegliere la connettività:
- Capannone A: forti interferenze WiFi, ma c'è copertura cellulare buona.
- Capannone B: all'aperto, pioggia frequente, lontano dalla città.
- Capannone C: in città con buona WiFi, ma budget molto limitato.

Per ogni capannone scegli la tecnologia di connettività più adatta tra: WiFi, 4G/LTE, LoRaWAN.
Motiva ogni scelta.
"""

# SENZA Chain-of-Thought
print("=" * 55)
print("SENZA CoT:")
print("=" * 55)
print(chiedi_claude(mio_problema, temperature=0))

print()

# CON Chain-of-Thought
print("=" * 55)
print("CON CoT:")
print("=" * 55)
problema_cot = mio_problema + "\nRagiona passo per passo: per ogni capannone elenca i vincoli, poi scegli la tecnologia e spiega perché."
print(chiedi_claude(problema_cot, temperature=0))

# Conclusione del gruppo:
# Il CoT aiuta anche sui problemi logici (non solo matematici): forzando il modello
# ad analizzare un vincolo alla volta, le scelte diventano più coerenti e ben motivate
# (es. A→4G per le interferenze WiFi, B→LoRaWAN per distanza/meteo, C→WiFi per budget).

---
## 📊 Preparate la presentazione (5 slide)

1. **Zero-shot vs Few-shot** — differenza pratica con i vostri risultati
2. **Quanti esempi servono?** — la vostra guida 1-shot / 3-shot / 5-shot
3. **Chain-of-Thought** — dimostrazione con il problema di calcolo
4. **CoT su problemi logici** — funziona anche lì?
5. **Quando usare quale tecnica** — la vostra guida pratica

---
*ITS Novitas 4.0 — AI Engineering Fundamentals | Marco Uras*